<a href="https://colab.research.google.com/github/KurniaYufi/sentiment-analysis-kematian-ali-khamenei/blob/main/scraping/tribun/tribun-content-scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scraping Artikel Tribunnews (Anti-Block Version)

## (1) Instalasi Library

In [ ]:
!pip install beautifulsoup4 requests pandas fake-useragent

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 7.0 MB/s eta 0:00:00


## (2) Impor Library

In [ ]:
import io
import time
import random
import urllib.parse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from google.colab import files

try:
    from fake_useragent import UserAgent
    ua = UserAgent()
    USE_FAKE_UA = True
    print("fake-useragent siap digunakan.")
except Exception:
    USE_FAKE_UA = False
    print("fake-useragent gagal, pakai daftar UA manual.")

✅ fake-useragent siap digunakan.


## (3) Konfigurasi Header & Session

In [ ]:
# Daftar User-Agent cadangan jika fake-useragent gagal
MANUAL_USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) Gecko/20100101 Firefox/125.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 14_4_1) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4.1 Safari/605.1.15',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
]

def get_random_ua():
    if USE_FAKE_UA:
        try:
            return ua.random
        except Exception:
            pass
    return random.choice(MANUAL_USER_AGENTS)

def get_headers(url):
    """Buat header yang menyerupai browser nyata."""
    parsed = urllib.parse.urlparse(url)
    origin  = f"{parsed.scheme}://{parsed.netloc}"
    return {
        'User-Agent'               : get_random_ua(),
        'Accept'                   : 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
        'Accept-Language'          : 'id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7',
        'Accept-Encoding'          : 'gzip, deflate, br',
        'Connection'               : 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Cache-Control'            : 'max-age=0',
        'Sec-Fetch-Dest'           : 'document',
        'Sec-Fetch-Mode'           : 'navigate',
        'Sec-Fetch-Site'           : 'none',
        'Sec-Fetch-User'           : '?1',
        'Referer'                  : 'https://www.google.com/',
        'Origin'                   : origin,
    }

def make_session():
    """Buat session baru dengan headers awal (simulasi buka halaman utama)."""
    s = requests.Session()
    try:
        s.get('https://www.tribunnews.com/', headers=get_headers('https://www.tribunnews.com/'), timeout=10)
        time.sleep(random.uniform(1.5, 3.0))
    except Exception:
        pass
    return s

print("Konfigurasi header selesai.")

✅ Konfigurasi header selesai.


## (4) Upload File CSV

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"File ter-upload: {filename}")

Saving Tugas 1A Kelompok 3 - Sheet8 (1).csv to Tugas 1A Kelompok 3 - Sheet8 (1).csv
File ter-upload: Tugas 1A Kelompok 3 - Sheet8 (1).csv


In [ ]:
urls = []

try:
    links_df = pd.read_csv(io.BytesIO(uploaded[filename]))
    print(f"Kolom ditemukan: {list(links_df.columns)}")

    if 'Link' in links_df.columns:
        urls = links_df['Link'].dropna().tolist()
        print(f"Berhasil memuat {len(urls)} URL.")
    else:
        print("Kolom 'Link' tidak ditemukan. Periksa header CSV kamu.")

except Exception as e:
    print(f"Gagal membaca file: {e}")

print("\nContoh 3 URL pertama:")
for u in urls[:3]:
    print(" ", u)

Kolom ditemukan: ['Link']
✅ Berhasil memuat 100 URL.

Contoh 3 URL pertama:
  https://aceh.tribunnews.com/news/1018596/duka-dan-amarah-di-pemakaman-komandan-al-irgc-warga-iran-bersumpah-melawan-as-sampai-akhir
  https://video.tribunnews.com/news/916067/media-iran-konfirmasi-kematian-ali-khamenei-beserta-anak-hingga-cucu-dalam-serangan-as-israel
  https://wartakota.tribunnews.com/news/883298/pemimpin-tertinggi-iran-tewas-ustaz-felix-siaw-officially-world-war-ketiga


## (5) Fungsi Scraping dengan Fallback Google Cache

In [ ]:
SKIP_PREFIXES = (
    "baca:", "tonton:", "video:", "tribun-video.com",
    "(*) ", "download tribunx", "simak breaking news",
    "ikuti kami di", "artikel ini telah", "follow ",
    "dapatkan pilihan", "bergabunglah",
)

def parse_article(soup, url):
    """Ekstrak judul dan konten dari objek BeautifulSoup."""
    # Judul
    title = 'No Title Found'
    for selector in [('h1', {'class_': 'f24'}), ('h1', {}), ('title', {})]:
        tag = soup.find(selector[0], **selector[1])
        if tag:
            title = tag.get_text(strip=True)
            break

    # Konten
    article_div = (
        soup.find('div', class_='side-article txt-article multi-fontsize') or
        soup.find('article') or
        soup.find('div', class_='article-content') or
        soup.find('div', class_='detail_text') or
        soup.find('div', class_='content')
    )

    paragraphs = article_div.find_all('p') if article_div else soup.find_all('p')

    filtered = [
        p.get_text(strip=True) for p in paragraphs
        if p.get_text(strip=True)
        and len(p.get_text(strip=True)) > 30
        and not p.get_text(strip=True).lower().startswith(SKIP_PREFIXES)
        and 'download tribunx untuk android' not in p.get_text(strip=True).lower()
    ]

    if not filtered and paragraphs:
        filtered = [p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)]

    return title, "\n".join(filtered)


def scrape_direct(session, url, max_retries=3):
    """Coba scrape langsung dari URL asli."""
    for attempt in range(max_retries):
        try:
            resp = session.get(url, headers=get_headers(url), timeout=15)
            if resp.status_code == 200:
                soup = BeautifulSoup(resp.content, 'html.parser')
                title, content = parse_article(soup, url)
                return title, content, None
            elif resp.status_code == 403:
                wait = (attempt + 1) * 3 + random.uniform(1, 3)
                print(f"    [403] Attempt {attempt+1}/{max_retries} — tunggu {wait:.1f}s...")
                time.sleep(wait)
            else:
                return None, None, f"Status {resp.status_code}"
        except requests.exceptions.Timeout:
            wait = (attempt + 1) * 2
            print(f"    [Timeout] Attempt {attempt+1}/{max_retries} — tunggu {wait}s...")
            time.sleep(wait)
        except Exception as e:
            return None, None, str(e)
    return None, None, f"Gagal setelah {max_retries} percobaan (403 terus)"


def scrape_google_cache(url):
    """Fallback: ambil dari Google Cache jika URL asli diblokir."""
    cache_url = f"https://webcache.googleusercontent.com/search?q=cache:{urllib.parse.quote(url)}"
    try:
        headers = get_headers(cache_url)
        headers['Referer'] = 'https://www.google.com/'
        resp = requests.get(cache_url, headers=headers, timeout=15)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.content, 'html.parser')
            title, content = parse_article(soup, url)
            return title, content, None
        else:
            return None, None, f"Cache status {resp.status_code}"
    except Exception as e:
        return None, None, f"Cache error: {e}"


def scrape_content(session, url):
    """Scrape dengan fallback otomatis ke Google Cache jika 403."""
    title, content, error = scrape_direct(session, url)

    if error and '403' in str(error):
        print(f"Coba Google Cache...")
        title, content, error = scrape_google_cache(url)
        if not error:
            print(f"Berhasil dari Google Cache.")

    return {"url": url, "title": title, "content": content, "error": error}


print("Fungsi scraping siap.")

✅ Fungsi scraping siap.


## (6) Proses Scraping

In [ ]:
data = []

if not urls:
    print("Tidak ada URL. Pastikan CSV sudah di-upload.")
else:
    total = len(urls)

    # Buat session awal (kunjungi halaman utama dulu)
    print("Membuat session awal...")
    session = make_session()
    print("Session siap.\n")

    for i, url in enumerate(urls):
        print(f"[{i+1}/{total}] {url}")

        # Refresh session setiap 20 artikel agar cookies tidak stale
        if i > 0 and i % 20 == 0:
            print("Refresh session...")
            session = make_session()

        result = scrape_content(session, url)

        if result['error']:
            print(f"Error: {result['error']}")
        else:
            judul = result['title'] or ''
            print(f"{judul[:70]}{'...' if len(judul) > 70 else ''}")

        data.append(result)

        # Delay acak: lebih panjang di awal, lebih pendek setelah sesi stabil
        if i < total - 1:
            delay = random.uniform(8, 15) if i < 5 else random.uniform(5, 10)
            print(f"  ⏳ Tunggu {delay:.1f}s...\n")
            time.sleep(delay)

    print(f"\n{'='*50}")
    print(f"Selesai. Total diproses: {len(data)}")
    sukses = sum(1 for d in data if not d['error'])
    print(f"   Sukses : {sukses}")
    print(f"   Gagal  : {len(data) - sukses}")
    print(f"{'='*50}")

# Buat DataFrame
df_article = pd.DataFrame(data)
display(df_article[['url', 'title', 'content']].head(5))

🔄 Membuat session awal...
✅ Session siap.

[1/100] https://aceh.tribunnews.com/news/1018596/duka-dan-amarah-di-pemakaman-komandan-al-irgc-warga-iran-bersumpah-melawan-as-sampai-akhir
  ✅ Duka dan Amarah di Pemakaman Komandan AL IRGC, Warga Iran Bersumpah Me...
  ⏳ Tunggu 10.6s...

[2/100] https://video.tribunnews.com/news/916067/media-iran-konfirmasi-kematian-ali-khamenei-beserta-anak-hingga-cucu-dalam-serangan-as-israel
  ✅ Media Iran Konfirmasi Kematian Ali Khamenei Beserta Anak hingga Cucu d...
  ⏳ Tunggu 11.2s...

[3/100] https://wartakota.tribunnews.com/news/883298/pemimpin-tertinggi-iran-tewas-ustaz-felix-siaw-officially-world-war-ketiga
  ✅ Pemimpin Tertinggi Iran Tewas, Ustaz Felix Siaw: Officially World War ...
  ⏳ Tunggu 13.7s...

[4/100] https://aceh.tribunnews.com/news/1018596/duka-dan-amarah-di-pemakaman-komandan-al-irgc-warga-iran-bersumpah-melawan-as-sampai-akhir
  ✅ Duka dan Amarah di Pemakaman Komandan AL IRGC, Warga Iran Bersumpah Me...
  ⏳ Tunggu 12.3s...

[5/100] ht

,url,title,content
0,https://aceh.tribunnews.com/news/1018596/duka-...,"Duka dan Amarah di Pemakaman Komandan AL IRGC,...",SERAMBINEWS.COM- Ribuan warga memadati jalanan...
1,https://video.tribunnews.com/news/916067/media...,Media Iran Konfirmasi Kematian Ali Khamenei Be...,Download aplikasi berita TribunX di Play Store...
2,https://wartakota.tribunnews.com/news/883298/p...,"Pemimpin Tertinggi Iran Tewas, Ustaz Felix Sia...",Ringkasan Berita:Presiden ASDonald Trumpmengum...
3,https://aceh.tribunnews.com/news/1018596/duka-...,"Duka dan Amarah di Pemakaman Komandan AL IRGC,...",SERAMBINEWS.COM- Ribuan warga memadati jalanan...
4,https://palu.tribunnews.com/news/180470/ditemu...,"Ditemukan di Bawah Reruntuhan, Ini Kronologi T...",TRIBUNPALU.COM- Jenazah Pemimpin TertinggiIran...


## (7) Ekspor Hasil ke CSV

In [ ]:
output_filename = 'tribun-content-scraping.csv'
df_article[['url', 'title', 'content']].to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"Hasil disimpan ke '{output_filename}'")

# Download otomatis
files.download(output_filename)